In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

In [3]:
RANDOM_STATE=42
df = pd.read_csv('data/train.csv')
df['PurchDate'] = pd.to_datetime(df['PurchDate'], format='%m/%d/%Y')
print(df.shape, df['IsBadBuy'].mean())

(72983, 34) 0.12298754504473644


In [4]:
dates = np.sort(df['PurchDate'].unique())
n = len(dates)
d1, d2 = dates[n // 3], dates[2 * n // 3]

train = df[df.PurchDate < d1].copy()
valid = df[(df.PurchDate >= d1) & (df.PurchDate < d2)].copy()
test = df[df.PurchDate >= d2].copy()

assert train.PurchDate.max() < valid.PurchDate.min() < valid.PurchDate.max() < test.PurchDate.min()
print(len(train), len(valid), len(test))
print(train.IsBadBuy.mean(), valid.IsBadBuy.mean(), test.IsBadBuy.mean())

23059 24104 25820
0.11305780823105946 0.13288250912711583 0.12261812548412084


In [5]:
def roc_auc(y_true, y_score):
    """AUC через ранги (Mann-Whitney U) с учётом ties"""
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score, float)
    pos, neg = (y_true == 1).sum(), (y_true == 0).sum()
    if pos == 0 or neg == 0:
        return np.nan
    order = np.argsort(y_score, kind='mergesort')
    s = y_score[order]
    ranks = np.empty(len(s), float)
    i = 0
    while i < len(s):
        j = i
        while j + 1 < len(s) and s[j + 1] == s[i]:
            j += 1
        ranks[i:j + 1] = (i + j) / 2 + 1
        i = j + 1
    r = np.empty(len(s), float)
    r[order] = ranks
    return (r[y_true == 1].sum() - pos * (pos + 1) / 2) / (pos * neg)

def gini(y_true, y_score):
    return 2 * roc_auc(y_true, y_score) - 1 

def confusion(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    return tp, fp, fn, tn

def precision_score_my(y_true, y_pred):
    tp, fp, _, _ = confusion(y_true, y_pred)
    return tp / (tp + fp) if tp + fp else 0.0

def recall_score_my(y_true, y_pred):
    tp, _, fn, _ = confusion(y_true, y_pred)
    return tp / (tp + fn) if tp + fn else 0.0

def f1_score_my(y_true, y_pred):
    p, r = precision_score_my(y_true, y_pred), recall_score_my(y_true, y_pred)
    return 2 * p * r / (p + r) if p + r else 0.0

def auc_pr(y_true, y_score):
    """Площадь под PR-кривой (трапеции по recall)."""
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score, float)
    order = np.argsort(-y_score, kind='mergesort')
    y = y_true[order]
    s = y_score[order]
    tp = np.cumsum(y == 1)
    fp = np.cumsum(y == 0)
    P = (y_true == 1).sum()
    keep = np.r_[s[1:] != s[:-1], True]
    prec = (tp / (tp + fp))[keep]
    rec = (tp / P)[keep]
    prec = np.r_[1.0, prec]
    rec = np.r_[0.0, rec]
    return np.trapezoid(prec, rec)

In [6]:
rng = np.random.default_rng(0)
yt = rng.integers(0, 2, 5000)
ys = rng.random(5000) * 0.3 + yt * 0.4
print(abs(gini(yt, ys) - abs(2 * roc_auc_score(yt, ys) - 1)) < 1e-9)

True


In [7]:
DROP = ['RefId', 'IsBadBuy', 'PurchDate']
CAT = ['Auction','Make','Model','Trim','SubModel','Color','Transmission','WheelType',
       'Nationality','Size','TopThreeAmericanName','PRIMEUNIT','AUCGUART','VNST',
       'BYRNO','VNZIP1','WheelTypeID']
NUM = ['VehYear','VehicleAge','VehOdo','VehBCost','WarrantyCost','IsOnlineSale',
       'MMRAcquisitionAuctionAveragePrice','MMRAcquisitionAuctionCleanPrice',
       'MMRAcquisitionRetailAveragePrice','MMRAcquisitonRetailCleanPrice',
       'MMRCurrentAuctionAveragePrice','MMRCurrentAuctionCleanPrice',
       'MMRCurrentRetailAveragePrice','MMRCurrentRetailCleanPrice']

class Prep:
    """Count-encoding категорий + медианная импутация + StandardScaler. Fit только на train."""
    def __init__(self, cat=CAT, num=NUM):
        self.cat, self.num = cat, num

    def _clean(self, df):
        d = df.copy()
        for c in self.num:
            d[c] = pd.to_numeric(d[c].replace('NULL', np.nan), errors='coerce')
        for c in self.cat:
            d[c] = d[c].astype(str).replace({'NULL': 'NA', 'nan': 'NA'})
        return d

    def fit(self, df):
        d = self._clean(df)
        self.counts_ = {c: d[c].value_counts() for c in self.cat}
        self.med_ = {c: d[c].median() for c in self.num}
        X = self._raw(d)
        self.cols_ = X.columns.tolist()
        self.mu_, self.sd_ = X.values.mean(0), X.values.std(0)
        self.sd_[self.sd_ == 0] = 1.0
        return self

    def _raw(self, d):
        out = {}
        for c in self.num:
            out[c] = d[c].fillna(self.med_[c])
        for c in self.cat:
            out[c + '_cnt'] = d[c].map(self.counts_[c]).fillna(0).astype(float)
        return pd.DataFrame(out, index=d.index)

    def transform(self, df):
        X = self._raw(self._clean(df))[self.cols_].values
        return (X - self.mu_) / self.sd_

prep = Prep().fit(train)
X_tr, y_tr = prep.transform(train), train.IsBadBuy.values
X_val, y_val= prep.transform(valid), valid.IsBadBuy.values

In [8]:
def report(name, model, X_tr, y_tr, X_val, y_val):
    model.fit(X_tr, y_tr)
    g = gini(y_val, model.predict_proba(X_val)[:, 1])
    print(f'{name:22s} valid Gini = {g:.4f}')
    return g

report('LogisticRegression', LogisticRegression(max_iter=2000, C=1.0), X_tr, y_tr, X_val, y_val)
report('GaussianNB', GaussianNB(), X_tr, y_tr, X_val, y_val)
report('KNN(k=100)', KNeighborsClassifier(100, n_jobs=-1), X_tr, y_tr, X_val, y_val)

LogisticRegression     valid Gini = 0.4324
GaussianNB             valid Gini = 0.4577
KNN(k=100)             valid Gini = 0.4519


np.float64(0.45188245500251156)

In [9]:
class MyLogisticRegression:
    """SGD, logloss, аналитический градиент: dL/dz = (sigma(z) - y)."""
    def __init__(self, lr=0.05, epochs=30, batch_size=64, l2=0.0, l1=0.0, class_weight=None, random_state=RANDOM_STATE):
        self.lr, self.epochs, self.bs = lr, epochs, batch_size
        self.l2, self.l1, self.cw, self.rs = l2, l1, class_weight, random_state

    @staticmethod
    def _sigmoid(z):
        return np.where(z >= 0, 1 / (1 + np.exp(-np.clip(z, -30, 30))), np.exp(np.clip(z, -30, 30)) / (1 + np.exp(np.clip(z, -30, 30))))
    
    def fit(self, X, y):
        X = np.asarray(X, float)
        y = np.asarray(y, float)
        n, m = X.shape
        rng = np.random.default_rng(self.rs)
        self.w = np.zeros(m)
        self.b = 0.0
        if self.cw == 'balanced':
            w1, w0, = n / (2 * y.sum()), n / (2 * (n - y.sum()))
        else:
            w1 = w0 = 1.0
        sw = np.where(y == 1, w1, w0)
        for ep in range(self.epochs):
            idx = rng.permutation(n)
            lr = self.lr / (1 + ep)                     # затухающий шаг
            for s in range(0, n, self.bs):
                b = idx[s:s + self.bs]
                Xb, yb, wb = X[b], y[b], sw[b]
                p = self._sigmoid(Xb @ self.w + self.b)
                err = (p - yb) * wb                     # градиент логлосса по z
                gw = Xb.T @ err / len(b) + self.l2 * self.w + self.l1 * np.sign(self.w)
                gb = err.mean()
                self.w -= lr * gw
                self.b -= lr * gb
        return self
    
    def predict_proba(self, X):
        p = self._sigmoid(np.asarray(X, float) @ self.w + self.b)
        return np.c_[1 - p, p]

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)

class MyGaussianNB:
    def fit(self, X, y):
        X = np.asarray(X, float)
        y = np.asarray(y, float)
        self.classes_ = np.unique(y)
        self.prior_, self.mu_, self.var_ = [], [], []
        for c in self.classes_:
            Xc = X[y == c]
            self.prior_.append(len(Xc) / len(X))
            self.mu_.append(Xc.mean(0))
            self.var_.append(Xc.var(0) + 1e-9)
        self.prior_ = np.array(self.prior_)
        self.mu_ = np.array(self.mu_)
        self.var_ = np.array(self.var_)
        return self

    def predict_proba(self, X):
        X = np.asarray(X, float)
        log = []
        for k in range(len(self.classes_)):
            l1 = -0.5 * (np.log(2 * np.pi * self.var_[k]) + (X - self.mu_[k]) ** 2 / self.var_[k]).sum(1)
            log.append(np.log(self.prior_[k]) + l1)
        log = np.array(log).T
        log -= log.max(1, keepdims=True)
        p = np.exp(log)
        return p / p.sum(1, keepdims=True)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)

class MyKNN:
    """Голосование по k ближайщим (евклид), батчами по 500 объектов"""
    def __init__(self, k=100):
        self.k = k

    def fit(self, X, y):
        self.X = np.asarray(X, float)
        self.y = np.asarray(y, float)
        return self

    def predict_proba(self, X):
        X = np.asarray(X, float)
        out = np.empty(len(X))
        xn = (self.X ** 2).sum(1)
        for s in range(0, len(X), 500):
            B = X[s:s + 500]
            d = xn[None, :] - 2 * B @ self.X.T + (B ** 2).sum(1)[:, None]
            nn = np.argpartition(d, self.k, axis=1)[:, :self.k]
            out[s:s + 500] = self.y[nn].mean(1)
        return np.c_[1 - out, out]

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)

In [10]:
print('MyLR ', gini(y_val, MyLogisticRegression(lr=0.1, epochs=30).fit(X_tr, y_tr).predict_proba(X_val)[:, 1]))
print('MyNB ', gini(y_val, MyGaussianNB().fit(X_tr, y_tr).predict_proba(X_val)[:, 1]))
print('MyKNN', gini(y_val, MyKNN(100).fit(X_tr, y_tr).predict_proba(X_val)[:, 1]))

MyLR  0.4536694202182918
MyNB  0.4576934005954032
MyKNN 0.45188245500251156


In [11]:
NEW = ['r_bcost_acq','r_bcost_cur','r_bcost_ret','r_clean_avg','r_ret_auc',
       'r_cur_acq','odo_per_year','cost_per_odo','warr_per_cost','gap_ret_bcost']

class Prep2(Prep):
    def __init__(self):
        super().__init__(cat=CAT, num=NUM + NEW)

    def _clean(self, df):
        d = df.copy()
        for c in NUM:
            d[c] = pd.to_numeric(d[c].replace('NULL', np.nan), errors='coerce')
        eps = 1.0
        d['r_bcost_acq']   = d.VehBCost / (d.MMRAcquisitionAuctionAveragePrice + eps)
        d['r_bcost_cur']   = d.VehBCost / (d.MMRCurrentAuctionAveragePrice + eps)
        d['r_bcost_ret']   = d.VehBCost / (d.MMRAcquisitionRetailAveragePrice + eps)
        d['r_clean_avg']   = d.MMRAcquisitionAuctionCleanPrice / (d.MMRAcquisitionAuctionAveragePrice + eps)
        d['r_ret_auc']     = d.MMRAcquisitionRetailAveragePrice / (d.MMRAcquisitionAuctionAveragePrice + eps)
        d['r_cur_acq']     = d.MMRCurrentAuctionAveragePrice / (d.MMRAcquisitionAuctionAveragePrice + eps)
        d['odo_per_year']  = d.VehOdo / (d.VehicleAge + 1)
        d['cost_per_odo']  = d.VehBCost / (d.VehOdo + eps)
        d['warr_per_cost'] = d.WarrantyCost / (d.VehBCost + eps)
        d['gap_ret_bcost'] = d.MMRAcquisitionRetailAveragePrice - d.VehBCost
        d[NEW] = d[NEW].replace([np.inf, -np.inf], np.nan)
        for c in self.cat:
            d[c] = d[c].astype(str).replace({'NULL': 'NA', 'nan': 'NA'})
        return d

    def fit(self, df):
        d = self._clean(df)
        self.gb_ = {}
        for cat, con, agg in [('Model','VehBCost','mean'), ('Model','VehOdo','mean'),
                              ('Make','VehBCost','mean'), ('SubModel','VehBCost','mean'),
                              ('VNST','VehBCost','mean'), ('Make','VehicleAge','mean'),
                              ('Model','WarrantyCost','mean')]:
            self.gb_[(cat, con, agg)] = d.groupby(cat)[con].agg(agg)
        self.gb_fallback_ = {con: d[con].median() for _, con, _ in self.gb_.keys()}
        return super().fit(df)

    def _raw(self, d):
        X = super()._raw(d)
        for (cat, con, agg), mp in self.gb_.items():
            name = f'{cat}_{con}_{agg}'
            X[name] = d[cat].map(mp).fillna(self.gb_fallback_[con]).astype(float)
            X[f'{con}_div_{name}'] = (d[con] / (X[name] + 1.0)).replace(
                [np.inf, -np.inf], np.nan).fillna(1.0)
        return X

prep2 = Prep2().fit(train)
X2_tr, X2_val = prep2.transform(train), prep2.transform(valid)

report('LR  + feats', LogisticRegression(max_iter=3000), X2_tr, y_tr, X2_val, y_val)
report('GNB + feats', GaussianNB(),                      X2_tr, y_tr, X2_val, y_val)
report('KNN + feats', KNeighborsClassifier(200, n_jobs=-1), X2_tr, y_tr, X2_val, y_val)

LR  + feats            valid Gini = 0.4280
GNB + feats            valid Gini = 0.4515
KNN + feats            valid Gini = 0.4651


np.float64(0.46507624223098465)

In [12]:
lr = LogisticRegression(max_iter=3000).fit(X2_tr, y_tr)
imp = pd.Series(np.abs(lr.coef_[0]), index=prep2.cols_).sort_values(ascending=False)

# (a) руками: топ-K по |coef|
for k in [10, 15, 20, 25, 30, 40]:
    cols = [prep2.cols_.index(c) for c in imp.index[:k]]
    g = gini(y_val, LogisticRegression(max_iter=3000).fit(X2_tr[:, cols], y_tr).predict_proba(X2_val[:, cols])[:, 1])
    print(f'top-{k:2d} feats: {g:.4f}')

# (b) L1
for C in [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]:
    m = LogisticRegression(l1_ratio=1, solver='liblinear', C=C, max_iter=5000).fit(X2_tr, y_tr)
    print(f'L1 C={C:<6} nnz={int((m.coef_ != 0).sum()):3d}'
          f' gini={gini(y_val, m.predict_proba(X2_val)[:, 1]):.4f}')

top-10 feats: 0.4582
top-15 feats: 0.4584
top-20 feats: 0.4488
top-25 feats: 0.4501
top-30 feats: 0.4438
top-40 feats: 0.4339
L1 C=0.003  nnz=  6 gini=0.4442
L1 C=0.01   nnz= 13 gini=0.4562
L1 C=0.03   nnz= 28 gini=0.4458
L1 C=0.1    nnz= 38 gini=0.4468
L1 C=0.3    nnz= 40 gini=0.4405
L1 C=1.0    nnz= 47 gini=0.4348


In [13]:
best = (-1, None)
for pen, sol in [(1, 'liblinear'), (0, 'lbfgs')]:
    for C in [0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0]:
        for cw in [None, 'balanced']:
            m = LogisticRegression(l1_ratio=pen, solver=sol, C=C, class_weight=cw,
                                   max_iter=5000).fit(X2_tr, y_tr)
            g = gini(y_val, m.predict_proba(X2_val)[:, 1])
            if g > best[0]:
                best = (g, m)
            print(f'{pen} C={C:<6} cw={str(cw):8s} gini={g:.4f}')
print('BEST valid gini:', round(best[0], 4), best[1])

1 C=0.003  cw=None     gini=0.4442
1 C=0.003  cw=balanced gini=0.4656
1 C=0.01   cw=None     gini=0.4562
1 C=0.01   cw=balanced gini=0.4570
1 C=0.03   cw=None     gini=0.4455
1 C=0.03   cw=balanced gini=0.4557
1 C=0.1    cw=None     gini=0.4467
1 C=0.1    cw=balanced gini=0.4510
1 C=0.3    cw=None     gini=0.4405
1 C=0.3    cw=balanced gini=0.4465
1 C=1.0    cw=None     gini=0.4346
1 C=1.0    cw=balanced gini=0.4468
1 C=3.0    cw=None     gini=0.4288
1 C=3.0    cw=balanced gini=0.4458
0 C=0.003  cw=None     gini=0.4491
0 C=0.003  cw=balanced gini=0.4568
0 C=0.01   cw=None     gini=0.4464
0 C=0.01   cw=balanced gini=0.4532
0 C=0.03   cw=None     gini=0.4435
0 C=0.03   cw=balanced gini=0.4501
0 C=0.1    cw=None     gini=0.4412
0 C=0.1    cw=balanced gini=0.4463
0 C=0.3    cw=None     gini=0.4320
0 C=0.3    cw=balanced gini=0.4454
0 C=1.0    cw=None     gini=0.4280
0 C=1.0    cw=balanced gini=0.4453
0 C=3.0    cw=None     gini=0.4314
0 C=3.0    cw=balanced gini=0.4457
BEST valid gini: 0.4

In [14]:
final = best[1]
X2_test, y_test = prep2.transform(test), test.IsBadBuy.values
for nm, X, y in [('train', X2_tr, y_tr), ('valid', X2_val, y_val), ('test', X2_test, y_test)]:
    print(f'{nm} Gini = {gini(y, final.predict_proba(X)[:, 1]):.4f}')

train Gini = 0.4695
valid Gini = 0.4656
test Gini = 0.3362


In [38]:
rows = []
models = {
    'LogReg(best)': final,
    'GaussianNB': GaussianNB().fit(X2_tr, y_tr),
    'KNN(200)': KNeighborsClassifier(200, n_jobs=-1).fit(X2_tr, y_tr),
    'MyLogReg': MyLogisticRegression(lr=0.003, epochs=200, batch_size=256, l2=0.0).fit(X2_tr, y_tr),
    'MyGaussianNB': MyGaussianNB().fit(X2_tr, y_tr),
    'MyKNN(200)': MyKNN(200).fit(X2_tr, y_tr),
}
base = y_test.mean()
for nm, m in models.items():
    p = m.predict_proba(X2_test)[:, 1]
    yp = (p >= 0.5).astype(int)
    rows.append([nm, gini(y_test, p), auc_pr(y_test, p),
                 precision_score_my(y_test, yp), recall_score_my(y_test, yp), f1_score_my(y_test, yp)])
res = pd.DataFrame(rows, columns=['model', 'Gini', 'AUC_PR', 'Precision', 'Recall', 'F1'])
print(res.round(4).to_string(index=False))
print('baseline AUC PR (доля класса 1) =', round(base, 4))

       model   Gini  AUC_PR  Precision  Recall     F1
LogReg(best) 0.3362  0.2055     0.1920  0.6121 0.2923
  GaussianNB 0.3014  0.1877     0.1465  0.1854 0.1637
    KNN(200) 0.4598  0.4074     0.7543  0.1794 0.2899
    MyLogReg 0.3002  0.1747     0.2120  0.3335 0.2592
MyGaussianNB 0.3012  0.1870     0.1465  0.1854 0.1637
  MyKNN(200) 0.4598  0.4074     0.7543  0.1794 0.2899
baseline AUC PR (доля класса 1) = 0.1226


### 11. Какую hard-label метрику предпочесть

Accuracy бесполезна: предсказав «все машины хорошие», получим ~88%. Нужна метрика на позитивном классе.


Я бы выбрал F1 по классу IsBadBuy=1 как основную сводную метрику (и для сравнения моделей — AUC PR, поскольку она не зависит от порога и не учитывает огромный TN, в отличие от ROC AUC).
